# Sentiment Analysis Model Training for Stock News

This notebook trains a dual-expertise sentiment model using:
- **Financial PhraseBank** (ET-style formal news)
- **Twitter Financial News** (social media style)

In [ ]:
# Install required packages (run if needed)
# !pip install torch transformers datasets scikit-learn matplotlib seaborn

## 1. Data Loading

In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

# Load ET-style formal news dataset
et_style_ds = load_dataset("takala/financial_phrasebank", "sentences_allagree", split="train", trust_remote_code=True)
print(f"Financial PhraseBank samples: {len(et_style_ds)}")
print(et_style_ds)

# Load Twitter-style social news dataset
tw_style_ds = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train", trust_remote_code=True)
print(f"\nTwitter Financial News samples: {len(tw_style_ds)}")
print(tw_style_ds)

## 2. Data Labelling

Standardize column names and labels across both datasets:
- Financial PhraseBank: 0=Negative, 1=Neutral, 2=Positive
- Twitter Financial News: 0=Bearish, 1=Bullish, 2=Neutral → Map to 0=Negative, 1=Neutral, 2=Positive

In [ ]:
# Standardize Financial PhraseBank
et_style_ds = et_style_ds.rename_column("label", "labels")

# Standardize Twitter dataset - map labels and rename columns
# Twitter: 0=Bearish→Negative(0), 1=Bullish→Positive(2), 2=Neutral→Neutral(1)
def map_twitter_labels(example):
    label_map = {0: 0, 1: 2, 2: 1}  # Bearish→0, Bullish→2, Neutral→1
    example["labels"] = label_map[example["label"]]
    example["sentence"] = example["text"]  # Standardize text column
    return example

tw_style_ds = tw_style_ds.map(map_twitter_labels)
tw_style_ds = tw_style_ds.remove_columns(["label", "text"])  # Remove old columns

# Verify label distributions
print("ET-style label distribution:", np.bincount([x["labels"] for x in et_style_ds]))
print("Twitter-style label distribution:", np.bincount([x["labels"] for x in tw_style_ds]))

# Label mapping reference
label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
print("\nLabel mapping:", label_names)

## 2.5 Handle Class Imbalance (Hybrid Approach)

**Strategy:**
1. **Keep all ET data** - High quality formal news, don't touch it
2. **Undersample Twitter Neutrals only** - Twitter has 6k+ neutrals (mostly spam/noise)
3. **Use Weighted Loss** - Apply class weights during training for additional balance

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import random

random.seed(42)

# --- HYBRID APPROACH ---
# Step 1: Keep ALL ET data (high quality)
print("ET Dataset (keeping all):")
et_labels = [x["labels"] for x in et_style_ds]
print(f"  Distribution: {dict(Counter(et_labels))}")
print(f"  Total: {len(et_style_ds)}")

# Step 2: Undersample ONLY Twitter Neutrals
print("\nTwitter Dataset (before):")
tw_labels = [x["labels"] for x in tw_style_ds]
print(f"  Distribution: {dict(Counter(tw_labels))}")

# Separate Twitter by class
tw_negative = [i for i, x in enumerate(tw_style_ds) if x["labels"] == 0]
tw_neutral = [i for i, x in enumerate(tw_style_ds) if x["labels"] == 1]
tw_positive = [i for i, x in enumerate(tw_style_ds) if x["labels"] == 2]

# Target: Reduce Twitter neutrals to match Twitter positive count
target_neutral = len(tw_positive)  # ~1923
tw_neutral_sampled = random.sample(tw_neutral, target_neutral)

print(f"\nUndersampling Twitter Neutrals: {len(tw_neutral)} → {len(tw_neutral_sampled)}")

# Create balanced Twitter indices
tw_balanced_indices = tw_negative + tw_neutral_sampled + tw_positive
tw_style_balanced = tw_style_ds.select(tw_balanced_indices)

print("\nTwitter Dataset (after):")
tw_balanced_labels = [x["labels"] for x in tw_style_balanced]
print(f"  Distribution: {dict(Counter(tw_balanced_labels))}")
print(f"  Total: {len(tw_style_balanced)}")

# Step 3: Combine ET (all) + Twitter (undersampled neutrals)
full_dataset = concatenate_datasets([et_style_ds, tw_style_balanced]).shuffle(seed=42)

print("\n--- Final Combined Dataset ---")
final_labels = [x["labels"] for x in full_dataset]
final_counts = Counter(final_labels)
print(f"Distribution: {dict(final_counts)}")
print(f"Total samples: {len(full_dataset)}")

# Step 4: Compute class weights for weighted loss
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=final_labels
)
class_weights_dict = {i: w for i, w in enumerate(class_weights)}
print(f"\nClass weights for training:")
print(f"  Negative: {class_weights_dict[0]:.3f}")
print(f"  Neutral:  {class_weights_dict[1]:.3f}")
print(f"  Positive: {class_weights_dict[2]:.3f}")

## 3. Data Exploration and Balance Visualization

In [ ]:
# Visualize the hybrid balanced dataset
print(f"Total samples: {len(full_dataset)}")

# Get label counts
labels = [x["labels"] for x in full_dataset]
label_counts = np.bincount(labels)

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Final distribution
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
axes[0].bar(label_names.values(), label_counts, color=colors)
axes[0].set_title('Hybrid Balanced Dataset')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

# Plot 2: Pie chart
axes[1].pie(label_counts, labels=label_names.values(), autopct='%1.1f%%', colors=colors, explode=(0.05, 0.05, 0.05))
axes[1].set_title('Sentiment Proportion')

# Plot 3: Original vs Hybrid comparison
original_counts = [1652, 7152, 2323]  # Original: ET + Twitter combined
x = np.arange(3)
width = 0.35
axes[2].bar(x - width/2, original_counts, width, label='Original', color='#ff9999', alpha=0.7)
axes[2].bar(x + width/2, label_counts, width, label='Hybrid', color='#66b3ff', alpha=0.7)
axes[2].set_xticks(x)
axes[2].set_xticklabels(label_names.values())
axes[2].set_title('Original vs Hybrid Approach')
axes[2].legend()
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

# Print improvement
print("\n--- Improvement Summary ---")
print(f"Original Neutral dominance: {original_counts[1]/sum(original_counts)*100:.1f}%")
print(f"Hybrid Neutral proportion: {label_counts[1]/sum(label_counts)*100:.1f}%")
print(f"Samples reduced: {sum(original_counts)} → {sum(label_counts)} ({sum(original_counts)-sum(label_counts)} removed)")

# Print sample texts
print("\n--- Sample Texts ---")
for label_id, label_name in label_names.items():
    sample = next(x for x in full_dataset if x["labels"] == label_id)
    print(f"\n{label_name}: {sample['sentence'][:100]}...")

## 4. Text Embedding (Tokenization)

In [ ]:
# Split dataset
full_dataset = full_dataset.train_test_split(test_size=0.2)
print(f"Train samples: {len(full_dataset['train'])}")
print(f"Test samples: {len(full_dataset['test'])}")

# Initialize tokenizer
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenization function
def tokenize_fn(examples):
    return tokenizer(
        examples["sentence"], 
        padding="max_length", 
        truncation=True,
        max_length=128
    )

# Apply tokenization
tokenized_ds = full_dataset.map(tokenize_fn, batched=True, remove_columns=["sentence"])
print(f"\nTokenized dataset features: {tokenized_ds['train'].features}")

# Show tokenization example
sample_text = full_dataset['train'][0]['sentence']
tokens = tokenizer(sample_text, truncation=True)
print(f"\n--- Tokenization Example ---")
print(f"Original: {sample_text[:80]}...")
print(f"Token IDs (first 20): {tokens['input_ids'][:20]}")
print(f"Decoded tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][:15])}")

## 5. Model Loading and Training

In [ ]:
# Load pre-trained FinBERT model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
print(f"Model: {model_name}")
print(f"Number of parameters: {model.num_parameters():,}")

# Training arguments
training_args = TrainingArguments(
    output_dir="./sentiment_expert_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# Metrics computation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions), 
        "f1": f1_score(labels, predictions, average='weighted')
    }

# Custom Trainer with class weights for additional robustness
from torch import nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(list(class_weights.values()), dtype=torch.float32)
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Move weights to same device as logits
        weights = self.class_weights.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# Initialize Weighted Trainer
trainer = WeightedTrainer(
    class_weights=class_weights_dict,
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics,
)

print("\nWeighted Trainer initialized with class weights:")
print(f"  Negative: {class_weights_dict[0]:.3f}")
print(f"  Neutral:  {class_weights_dict[1]:.3f}")
print(f"  Positive: {class_weights_dict[2]:.3f}")
print("\nReady to train!")

In [ ]:
# Start Training
print("Starting training...")
trainer.train()

In [ ]:
# Evaluate final model
eval_results = trainer.evaluate()
print("\n--- Final Evaluation Results ---")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

# Save the model
model.save_pretrained("./final_sentiment_model")
tokenizer.save_pretrained("./final_sentiment_model")
print("\nModel saved to ./final_sentiment_model")